# Semaine 4 — profilage GPU, ablation, test final, figures (Colab T4)

**Avant de lancer :** `Exécution > Modifier le type d'exécution > GPU T4`, puis `Exécution > Tout exécuter`
(ou cellule par cellule avec Maj+Entrée). Garde l'onglet ouvert : Colab coupe une session inactive.

Le notebook est en **deux parties**, dans l'ordre demandé par l'équipe :
- **Partie A (~40 min)** : profilage GPU de `optimized` → ablation (une optimisation désactivée à la fois, 3 seeds).
- **Partie B (~40 min)** : seulement si la config est figée → évaluation **unique** sur le test → benchmark de bout en bout CPU vs GPU → figures.

À la fin, un fichier `week4_results.zip` est téléchargé. Il contient tous les résultats à mettre dans `results/` sur GitHub.

## 0. Préparation (GPU, code, données)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| CUDA dispo :", torch.cuda.is_available())
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"

Sauvegarde sur Google Drive (recommandé) : si Colab coupe, les résultats déjà calculés sont conservés et
le notebook reprend là où il s'était arrêté (les runs déjà faits sont sautés). Autorise l'accès quand Colab le demande.

In [ ]:
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/gpu_dr_week4"
else:
    SAVE = "/content/week4_save"
os.makedirs(SAVE, exist_ok=True)
print("sauvegarde ->", SAVE)

**Code** : on clone `main` (semaines 1-3), puis on ajoute les fichiers de la semaine 4.
Quand la cellule le demande, choisis le fichier **`week4_amina.zip`** (donné par Claude).
Si la branche `week4-amina` existe déjà sur GitHub avec ces fichiers, mets `FROM_BRANCH = True` et rien n'est demandé.

In [ ]:
FROM_BRANCH = False
REPO = "https://github.com/zeynabbelkasmi/gpu-dr-project"
%cd /content
if not os.path.isdir("/content/gpu-dr-project"):
    !git clone -q {REPO} /content/gpu-dr-project
%cd /content/gpu-dr-project
if FROM_BRANCH:
    !git fetch -q origin week4-amina && git checkout -q week4-amina
elif not os.path.exists("scripts/week4_ablation.py"):
    from google.colab import files
    up = files.upload()                                   # choisir week4_amina.zip
    name = next(iter(up))
    !unzip -o -q "{name}" -d /content/gpu-dr-project && rm -f "{name}"
!git log --oneline -1
!ls scripts/week4_*.py benchmarks/profile_gpu.py configs/exp_*_off.yaml
# reprise après coupure : on récupère ce qui est déjà sur Drive
if os.path.isdir(f"{SAVE}/week4"):
    !mkdir -p results && cp -rn {SAVE}/week4 results/ && echo "résultats précédents restaurés"

In [ ]:
!pip -q install -r requirements.txt 2>&1 | tail -2
os.environ["FDR26_DATA"] = "/content/data"
!python scripts/download_data.py 2>&1 | tail -3

In [ ]:
# Table 1 (environnement de la semaine 4) + tests (équivalence CPU/GPU, prefetcher)
!python -c "import sys; sys.path.insert(0,'scripts'); import record_env; record_env.main('results/week4/environment.json')"
!cat results/week4/environment.json
!python -m pytest tests -q 2>&1 | tail -4

In [ ]:
import shutil
def save():
    """Copie results/week4 sur Drive (reprise possible si Colab coupe)."""
    if os.path.isdir("results/week4"):
        shutil.copytree("results/week4", f"{SAVE}/week4", dirs_exist_ok=True)
        print("sauvegardé dans", SAVE)

# Partie A
## A1. Profilage GPU de la config finale (A7.2-A7.4) — ~3 min
Où partent les 4,5 s d'une époque ? Le script mesure : le temps d'un pas d'entraînement, la part du temps
où le GPU est **occupé vs inactif**, les copies CPU→GPU, le temps GPU par catégorie de kernels
(convolutions, batch norm, optimiseur…), les phases forward/backward/optimizer, et la part de la validation.

In [ ]:
!python benchmarks/profile_gpu.py --config configs/optimized.yaml
from IPython.display import Image, display
display(Image("results/week4/profile_optimized/timeline.png"))

Même profilage **sans** les données résidentes sur le GPU (chemin DataLoader) : pour voir ce que la semaine 3 a supprimé (copies, attente du GPU).

In [ ]:
!python benchmarks/profile_gpu.py --config configs/exp_resident_off.yaml --no-epochs
display(Image("results/week4/profile_exp_resident_off/timeline.png"))
save()

## A2. Ablation (A7.6) — ~30 min
Depuis `optimized.yaml`, on désactive **une seule** optimisation à la fois (+ une combinaison définie :
tout le pipeline d'entrée GPU), 3 seeds chacune, même GPU, même split, 5 époques.
`full` (= la config finale) est relancée ici pour avoir une référence sur le même GPU.

In [ ]:
!python scripts/week4_ablation.py --seeds 42,43,44 2>&1 | grep -v "^epoch"
save()

In [ ]:
# Résumé de la partie A
import json, pandas as pd
s = json.load(open("results/week4/profile_optimized/summary.json"))
p = s["profiler"]
print(f"pas d'entraînement : {s['step_ms_unsynchronized']:.1f} ms | GPU inactif : {100*p['gpu_idle_fraction']:.1f} %")
print("copies CPU->GPU :", {k: f"{v['bytes_per_step']/2**20:.2f} MiB/pas" for k, v in p["memcpy"].items()} or "aucune")
print("temps GPU par catégorie :", {k: f"{100*v:.0f} %" for k, v in p["kernel_categories_share"].items()})
print("phases :", {k: f"{v:.1f} ms" for k, v in s["phases_ms_synchronized"].items()})
if s.get("epoch"): print("validation =", f"{100*s['epoch']['val_share']:.0f} % de l'époque")
a = pd.read_csv("results/week4/ablation.csv", index_col=0)
print(a[["n", "steady_epoch_s_mean", "slowdown_vs_full", "val_f1_at_best_mean", "val_f1_at_best_std"]].round(3))

## Décision : figer la config ?
Règle de l'équipe : le test n'est utilisé **qu'une fois la config figée**. La cellule suivante vérifie le profil :
- si le GPU est inactif **moins de 10 %** du temps, le pipeline est limité par le calcul du GPU lui-même
  (convolutions) : il n'y a plus de goulot côté pipeline à optimiser → on fige `configs/optimized.yaml` et on continue.
- sinon, **arrête-toi ici** : télécharge le zip (dernière cellule) et donne-le à Claude pour décider quoi optimiser.

In [ ]:
FORCE_FREEZE = False   # mettre True seulement si l'équipe a décidé de figer malgré un GPU inactif > 10 %
idle = json.load(open("results/week4/profile_optimized/summary.json"))["profiler"]["gpu_idle_fraction"]
FROZEN = idle < 0.10 or FORCE_FREEZE
print(f"GPU inactif {100*idle:.1f} % ->", "config FIGÉE, on continue (partie B)" if FROZEN
      else "NE PAS continuer : aller directement à la dernière cellule (zip) et demander à Claude")

# Partie B (seulement si `FROZEN = True`)
## B1. Évaluation finale sur le test (A7.8) — une seule fois, ~6 min
Entraîne la baseline et la config finale (seeds 42, 43, 44), vérifie sur la **validation** que l'évaluation
reproduit la précision du checkpoint, puis évalue **une fois** sur le test. Un fichier verrou
`results/week4/TEST_SET_USED.json` empêche de relancer par erreur.

In [ ]:
assert FROZEN, "config non figée : ne pas toucher au test"
if not os.path.exists("results/week4/TEST_SET_USED.json"):
    !python scripts/week4_test_eval.py 2>&1 | grep -v "^epoch"
else:
    print("test déjà évalué :", open("results/week4/TEST_SET_USED.json").read())
save()

## B2. Benchmark de bout en bout (A7.1, A7.5) — ~30 min (la baseline CPU est lente)
Baseline CPU et config finale GPU **sur la même machine** : un run de chauffe jeté, puis répétitions
(2 époques chacune ; le total pour 5 époques = setup + époque froide + 4 × époque stable).

In [ ]:
!python benchmarks/benchmark_end_to_end.py --only optimized_gpu --epochs 2 --repeats 3 2>&1 | grep -v "^epoch"
save()
!python benchmarks/benchmark_end_to_end.py --only baseline_cpu --epochs 2 --cpu-repeats 2 2>&1 | grep -v "^epoch"
save()

## B3. Figures et tableaux du rapport (A8.1-A8.6)

In [ ]:
!python scripts/make_figures.py
for f in ["fig1_baseline_breakdown", "fig2_speedup", "fig3_accuracy_vs_time", "fig4_ablation",
          "fig5_resources", "fig6_confusion"]:
    if os.path.exists(f"results/figures/{f}.png"):
        display(Image(f"results/figures/{f}.png"))

# Fin : télécharger les résultats
`week4_results.zip` contient `results/week4/` (sans les gros `trace.json`), les figures et `results/tables.md`.
Les `trace.json` (à ouvrir sur https://ui.perfetto.dev pour une capture d'écran) restent sur ton Drive.

In [ ]:
save()
!cp -r results/figures results/tables.md {SAVE}/ 2>/dev/null; true
!rm -f /content/week4_results.zip
!zip -q -r /content/week4_results.zip results/week4 results/figures results/tables.md -x "*/trace.json" "*.npy"
!ls -lh /content/week4_results.zip
from google.colab import files
files.download("/content/week4_results.zip")